使用create_cistarget_motif_databases.py构建cisTarget数据库前的准备工作： 生成create_cistarget_motif_databases.py -f参数对应的fasta文件

In [1]:
%%bash

mkdir -p "./fasta"
wget -O "./fasta/Apis_mellifera.fasta.gz" "https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/dna/Apis_mellifera.Amel_HAv3.1.dna.toplevel.fa.gz"
gunzip "./fasta/Apis_mellifera.fasta.gz"

--2026-08-12 16:15:40--  https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/dna/Apis_mellifera.Amel_HAv3.1.dna.toplevel.fa.gz
Resolving ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)... 193.62.193.161
Connecting to ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)|193.62.193.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 66731516 (64M) [application/x-gzip]
Saving to: './fasta/Apis_mellifera.fasta.gz'

     0K .......... .......... .......... .......... ..........  0% 84.0K 12m55s
    50K .......... .......... .......... .......... ..........  0%  126M 6m28s
   100K .......... .......... .......... .......... ..........  0%  167K 6m28s
   150K .......... .......... .......... .......... ..........  0%  140M 4m51s
   200K .......... .......... .......... .......... ..........  0%  169K 5m10s
   250K .......... .......... .......... .......... ..........  0% 51.0M 4m18s
   300K .......... .......... .........

Mu et al., 2025取的区域是: 基因起始位置(非狭义的TSS)上游5kb, 下游2kb

In [ ]:
%%bash

awk '$0 !~ /^#/ && $3=="gene"' "./gtf/Apis_mellifera.gtf" > "./gtf/Apis_mellifera_genes.gtf" # 仅保留类型为gene的行, 且去掉注释行
bedtools sort -i "./gtf/Apis_mellifera_genes.gtf" > "./gtf/Apis_mellifera_genes_sorted.gtf" # 排序

In [ ]:
%%bash
#! 根据HY以往的操作记录(.subproject/01_choose_gene_in_cistarget/01_choose_gene_in_cistarget.py)和asertlab提供的feather，还需要去掉全部线粒体基因，使用剩余基因建库
# CM009947.2 是线粒体对应的染色体名称
awk '$1 != "CM009947.2"' "./gtf/Apis_mellifera_genes_sorted.gtf" > "./gtf/Apis_mellifera_genes_sorted_nomt.gtf"

In [ ]:
%%bash

wc -l "./gtf/Apis_mellifera_genes_sorted.gtf"
wc -l "./gtf/Apis_mellifera_genes_sorted_nomt.gtf"
# 37个基因，未出错

12398 ./gtf/Apis_mellifera_genes_sorted.gtf
12361 ./gtf/Apis_mellifera_genes_sorted_nomt.gtf


# 开始提取上游5kb和下游2kb

### 读取GTF

In [26]:
import pyranges as pr

am_gtf = pr.read_gtf("./gtf/Apis_mellifera_genes_sorted_nomt.gtf").df
am_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_source,gene_biotype,gene_name
0,CM009931.2,RefSeq,gene,10791,17180,.,+,.,LOC551555,RefSeq,protein_coding,NaN
1,CM009931.2,RefSeq,gene,23612,26208,.,+,.,GeneID_409940,RefSeq,protein_coding,Rfwd3
2,CM009931.2,RefSeq,gene,29939,39215,.,+,.,LOC551448,RefSeq,protein_coding,NaN
3,CM009931.2,RefSeq,gene,42764,45255,.,+,.,LOC408567,RefSeq,protein_coding,NaN
4,CM009931.2,RefSeq,gene,45527,79393,.,+,.,LOC107964061,RefSeq,protein_coding,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
12356,QIUM02000210.1,RefSeq,gene,457,3977,.,-,.,LOC113219357,RefSeq,pseudogene,NaN
12357,QIUM02000210.1,RefSeq,gene,5407,5560,.,-,.,LOC113219356,RefSeq,rRNA,NaN
12358,QIUM02000211.1,RefSeq,gene,0,3962,.,+,.,LOC113219387,RefSeq,rRNA,NaN
12359,QIUM02000217.1,RefSeq,gene,556,1523,.,-,.,LOC113219358,RefSeq,protein_coding,NaN


### 添加各染色体长度(防止下游2kb超过染色体长度)

In [6]:
%%bash

wget -O "./fasta/Apis_mellifera.fai" "https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/dna_index/Apis_mellifera.Amel_HAv3.1.dna.toplevel.fa.gz.fai"

--2026-08-14 21:26:02--  https://ftp.ensemblgenomes.ebi.ac.uk/pub/metazoa/current/fasta/apis_mellifera/dna_index/Apis_mellifera.Amel_HAv3.1.dna.toplevel.fa.gz.fai


Resolving ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)... 193.62.193.161
Connecting to ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)|193.62.193.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6155 (6.0K) [application/x-gzip]
Saving to: './fasta/Apis_mellifera.fai'

     0K ......                                                100%  649M=0s

2026-08-14 21:26:04 (649 MB/s) - './fasta/Apis_mellifera.fai' saved [6155/6155]



In [13]:
import pandas as pd

am_genome_length = pd.read_csv("./fasta/Apis_mellifera.fai", header=None, sep="\t")[[0, 1]].set_index(0)[1]
am_genome_length

0
QIUM02000069.1      486754
QIUM02000070.1      311923
QIUM02000071.1      135562
QIUM02000072.1      120686
QIUM02000073.1      108711
                    ...   
CM009943.2        11279722
CM009944.2        10670842
CM009945.2         9534514
CM009946.2         7238532
CM009947.2           16343
Name: 1, Length: 177, dtype: int64

In [27]:
am_gtf["chrom_length"] = am_gtf["Chromosome"].map(am_genome_length)
am_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_source,gene_biotype,gene_name,chrom_length
0,CM009931.2,RefSeq,gene,10791,17180,.,+,.,LOC551555,RefSeq,protein_coding,NaN,27754200
1,CM009931.2,RefSeq,gene,23612,26208,.,+,.,GeneID_409940,RefSeq,protein_coding,Rfwd3,27754200
2,CM009931.2,RefSeq,gene,29939,39215,.,+,.,LOC551448,RefSeq,protein_coding,NaN,27754200
3,CM009931.2,RefSeq,gene,42764,45255,.,+,.,LOC408567,RefSeq,protein_coding,NaN,27754200
4,CM009931.2,RefSeq,gene,45527,79393,.,+,.,LOC107964061,RefSeq,protein_coding,NaN,27754200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12356,QIUM02000210.1,RefSeq,gene,457,3977,.,-,.,LOC113219357,RefSeq,pseudogene,NaN,5948
12357,QIUM02000210.1,RefSeq,gene,5407,5560,.,-,.,LOC113219356,RefSeq,rRNA,NaN,5948
12358,QIUM02000211.1,RefSeq,gene,0,3962,.,+,.,LOC113219387,RefSeq,rRNA,NaN,5439
12359,QIUM02000217.1,RefSeq,gene,556,1523,.,-,.,LOC113219358,RefSeq,protein_coding,NaN,4502


In [33]:
sum(am_gtf["chrom_length"].isna()) # 追加无误，继续

0

In [29]:
import numpy as np

plus = am_gtf["Strand"] == "+"

am_gtf["Neo_Start"] = np.where(plus, am_gtf["Start"] - 5000, am_gtf["End"] - 2001)
am_gtf["Neo_End"]   = np.where(plus, am_gtf["Start"] + 2001, am_gtf["End"] + 5000)
am_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_source,gene_biotype,gene_name,chrom_length,Neo_Start,Neo_End
0,CM009931.2,RefSeq,gene,10791,17180,.,+,.,LOC551555,RefSeq,protein_coding,NaN,27754200,5791,12792
1,CM009931.2,RefSeq,gene,23612,26208,.,+,.,GeneID_409940,RefSeq,protein_coding,Rfwd3,27754200,18612,25613
2,CM009931.2,RefSeq,gene,29939,39215,.,+,.,LOC551448,RefSeq,protein_coding,NaN,27754200,24939,31940
3,CM009931.2,RefSeq,gene,42764,45255,.,+,.,LOC408567,RefSeq,protein_coding,NaN,27754200,37764,44765
4,CM009931.2,RefSeq,gene,45527,79393,.,+,.,LOC107964061,RefSeq,protein_coding,NaN,27754200,40527,47528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12356,QIUM02000210.1,RefSeq,gene,457,3977,.,-,.,LOC113219357,RefSeq,pseudogene,NaN,5948,1976,8977
12357,QIUM02000210.1,RefSeq,gene,5407,5560,.,-,.,LOC113219356,RefSeq,rRNA,NaN,5948,3559,10560
12358,QIUM02000211.1,RefSeq,gene,0,3962,.,+,.,LOC113219387,RefSeq,rRNA,NaN,5439,-5000,2001
12359,QIUM02000217.1,RefSeq,gene,556,1523,.,-,.,LOC113219358,RefSeq,protein_coding,NaN,4502,-478,6523


#### 如果Neo_Start为负数，将其变成0，如果Neo_End超出了同行chrom_length，将其替换为chrom_length

In [30]:
am_gtf["Neo_Start"] = am_gtf["Neo_Start"].clip(lower=0)
am_gtf["Neo_End"] = am_gtf[["Neo_End", "chrom_length"]].min(axis=1)
am_gtf

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_source,gene_biotype,gene_name,chrom_length,Neo_Start,Neo_End
0,CM009931.2,RefSeq,gene,10791,17180,.,+,.,LOC551555,RefSeq,protein_coding,NaN,27754200,5791,12792
1,CM009931.2,RefSeq,gene,23612,26208,.,+,.,GeneID_409940,RefSeq,protein_coding,Rfwd3,27754200,18612,25613
2,CM009931.2,RefSeq,gene,29939,39215,.,+,.,LOC551448,RefSeq,protein_coding,NaN,27754200,24939,31940
3,CM009931.2,RefSeq,gene,42764,45255,.,+,.,LOC408567,RefSeq,protein_coding,NaN,27754200,37764,44765
4,CM009931.2,RefSeq,gene,45527,79393,.,+,.,LOC107964061,RefSeq,protein_coding,NaN,27754200,40527,47528
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12356,QIUM02000210.1,RefSeq,gene,457,3977,.,-,.,LOC113219357,RefSeq,pseudogene,NaN,5948,1976,5948
12357,QIUM02000210.1,RefSeq,gene,5407,5560,.,-,.,LOC113219356,RefSeq,rRNA,NaN,5948,3559,5948
12358,QIUM02000211.1,RefSeq,gene,0,3962,.,+,.,LOC113219387,RefSeq,rRNA,NaN,5439,0,2001
12359,QIUM02000217.1,RefSeq,gene,556,1523,.,-,.,LOC113219358,RefSeq,protein_coding,NaN,4502,0,4502


### 因为需要使用bedtools getfasta -s，所以必须保留Strand列：

In [35]:
am_gtf_4_bed = am_gtf[["Chromosome", "Neo_Start", "Neo_End", "gene_id", "Score", "Strand"]]
am_gtf_4_bed

,Chromosome,Neo_Start,Neo_End,gene_id,Score,Strand
0,CM009931.2,5791,12792,LOC551555,.,+
1,CM009931.2,18612,25613,GeneID_409940,.,+
2,CM009931.2,24939,31940,LOC551448,.,+
3,CM009931.2,37764,44765,LOC408567,.,+
4,CM009931.2,40527,47528,LOC107964061,.,+
...,...,...,...,...,...,...
12356,QIUM02000210.1,1976,5948,LOC113219357,.,-
12357,QIUM02000210.1,3559,5948,LOC113219356,.,-
12358,QIUM02000211.1,0,2001,LOC113219387,.,+
12359,QIUM02000217.1,0,4502,LOC113219358,.,-


In [36]:
am_gtf_4_bed.to_csv("./metadata/Amel_4_cisTarget.bed", sep="\t", index=False, header=False)

# 根据bed文件生成fasta

In [9]:
%%bash

bedtools getfasta \
    -fi "./fasta/Apis_mellifera.fasta" \
    -bed "./metadata/Amel_4_cisTarget.bed" \
    -name \
    -s \
    -fo "./fasta/Apis_mellifera_4_cisTarget.fasta"

In [ ]:
import pyfaidx

Am4cis = pyfaidx.Fasta("./fasta/Apis_mellifera_4_cisTarget.fasta")
# 检查Apis_mellifera_4_cisTarget.fasta.fai，确定长度是7001以保证bedtools没取错:

In [12]:
%%bash

head "./fasta/Apis_mellifera_4_cisTarget.fasta.fai"

LOC551555::CM009931.2:5791-12792(+)	7001	37	7001	7002
GeneID_409940::CM009931.2:18612-25613(+)	7001	7081	7001	7002
LOC551448::CM009931.2:24939-31940(+)	7001	14121	7001	7002
LOC408567::CM009931.2:37764-44765(+)	7001	21161	7001	7002
LOC107964061::CM009931.2:40527-47528(+)	7001	28204	7001	7002
LOC113219112::CM009931.2:86243-93244(+)	7001	35247	7001	7002
LOC726450::CM009931.2:86681-93682(+)	7001	42287	7001	7002
LOC408565::CM009931.2:112007-119008(+)	7001	49329	7001	7002
LOC113219398::CM009931.2:121476-128477(+)	7001	56374	7001	7002
GeneID_104796173::CM009931.2:134121-141122(+)	7001	63423	7001	7002


# 获取cBust，即将开始构建cisTarget数据库

In [ ]:
%%bash

cd ~/.local/bin
wget https://resources.aertslab.org/cistarget/programs/cbust
chmod +x cbust